# 19. 코드공유 참고 - age, mean_working 빼보기

데이콘 코드공유에 올라온 8등(XGBoost) 풀이 보니까 파생변수 하나도 없이 원본 컬럼만 쓰는데
그중에서도 age, mean_working을 아예 뺐음. 우리 데이터로도 확인해봄.

In [1]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42

train = pd.read_csv('../data/train.csv')
train = train.drop_duplicates(subset=[c for c in train.columns if c != 'ID']).reset_index(drop=True)
for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
train['edu_level'] = train['edu_level'].fillna('Unknown')
train['bmi'] = train['weight'] / ((train['height'] / 100) ** 2)

RAW14 = ['gender', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
         'diastolic_blood_pressure', 'glucose', 'bone_density', 'activity',
         'smoke_status', 'medical_history', 'family_medical_history',
         'sleep_pattern', 'edu_level']
CAT_COLS = ['gender', 'activity', 'smoke_status', 'medical_history',
            'family_medical_history', 'sleep_pattern', 'edu_level']

with open('optuna_round2_best_params.json') as f:
    tuned_params = json.load(f)

def encode(df, cols):
    d = df[cols + ['stress_score']].copy()
    for c in CAT_COLS:
        if c in d.columns:
            d[c] = LabelEncoder().fit_transform(d[c])
    return d

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def run_cv(cols, label):
    d = encode(train, cols)
    x = d.drop(columns=['stress_score']); y = d['stress_score']
    oof = np.zeros(len(x))
    for tr_idx, va_idx in kf.split(x):
        m = LGBMRegressor(**tuned_params)
        m.fit(x.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(x.iloc[va_idx], y.iloc[va_idx])],
              callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[va_idx] = m.predict(x.iloc[va_idx])
    mae = mean_absolute_error(y, oof)
    print(label, mae)
    return mae

run_cv(RAW14, 'raw14만')
run_cv(RAW14 + ['age', 'mean_working'], 'raw14 + age + mean_working')
run_cv(RAW14 + ['bmi'], 'raw14 + bmi')
run_cv(RAW14 + ['age'], 'raw14 + age만')
run_cv(RAW14 + ['mean_working'], 'raw14 + mean_working만')

raw14만 0.1629
raw14 + age + mean_working 0.1735
raw14 + bmi 0.1625
raw14 + age만 0.1649
raw14 + mean_working만 0.1731


mean_working 하나가 CV를 0.01 넘게 깎아먹고 있음. age도 소폭 해로움.
결측 34%가 age<19 or age>65일 때만 생기는 거라 (#15~17에서 이미 확인) 0으로 채운 게
틀린 건 아닌데, 변수 자체가 노이즈였던 듯. working_age_ratio, is_overworking 같은
파생변수도 다 이 변수에서 나온 거라 같이 빼야 함.

우리가 예전에 잘 나왔던 파생변수들 다시 얹어보면?

In [2]:
train['pulse_pressure'] = train['systolic_blood_pressure'] - train['diastolic_blood_pressure']
train['map'] = train['diastolic_blood_pressure'] + (train['pulse_pressure'] / 3)
train['cardio_metabolic_load'] = train['map'] * train['bmi']
train['glucose_chol_ratio'] = train['glucose'] / (train['cholesterol'] + 1)
train['is_hypertension'] = ((train['systolic_blood_pressure'] >= 140) | (train['diastolic_blood_pressure'] >= 90)).astype(int)

GOOD = ['bmi', 'pulse_pressure', 'map', 'cardio_metabolic_load', 'glucose_chol_ratio', 'is_hypertension']
run_cv(RAW14 + GOOD, '전부 다 얹기')
run_cv(RAW14 + ['bmi', 'cardio_metabolic_load'], 'bmi + cardio_metabolic_load만')

전부 다 얹기 0.1645
bmi + cardio_metabolic_load만 0.1635


raw14+bmi 단독(0.1625)보다 다 나쁨. 여기서는 더 넣을수록 손해.

타겟인코딩까지 추가

In [3]:
RAW14_BMI = RAW14 + ['bmi']

def run_te(use_te, label):
    oof = np.zeros(len(train))
    for tr_idx, va_idx in kf.split(train):
        tr_df = train[RAW14_BMI + ['stress_score']].iloc[tr_idx].copy()
        va_df = train[RAW14_BMI + ['stress_score']].iloc[va_idx].copy()
        if use_te:
            combo_tr = tr_df['medical_history'] + '_' + tr_df['family_medical_history']
            gmean = tr_df['stress_score'].mean()
            means = tr_df.groupby(combo_tr)['stress_score'].mean()
            tr_df['disease_combo_te'] = combo_tr.map(means).fillna(gmean)
            combo_va = va_df['medical_history'] + '_' + va_df['family_medical_history']
            va_df['disease_combo_te'] = combo_va.map(means).fillna(gmean)
            for c in ['gender', 'activity', 'smoke_status', 'sleep_pattern', 'edu_level']:
                le = LabelEncoder().fit(tr_df[c])
                tr_df[c] = le.transform(tr_df[c])
                unseen = [l for l in np.unique(va_df[c]) if l not in le.classes_]
                if unseen:
                    le.classes_ = np.append(le.classes_, unseen)
                va_df[c] = le.transform(va_df[c])
            tr_df = tr_df.drop(columns=['medical_history', 'family_medical_history'])
            va_df = va_df.drop(columns=['medical_history', 'family_medical_history'])
        else:
            for c in CAT_COLS:
                le = LabelEncoder().fit(tr_df[c])
                tr_df[c] = le.transform(tr_df[c])
                unseen = [l for l in np.unique(va_df[c]) if l not in le.classes_]
                if unseen:
                    le.classes_ = np.append(le.classes_, unseen)
                va_df[c] = le.transform(va_df[c])
        x_tr = tr_df.drop(columns=['stress_score']); y_tr = tr_df['stress_score']
        x_va = va_df.drop(columns=['stress_score']); y_va = va_df['stress_score']
        m = LGBMRegressor(**tuned_params)
        m.fit(x_tr, y_tr, eval_set=[(x_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[va_idx] = m.predict(x_va)
    mae = mean_absolute_error(train['stress_score'], oof)
    print(label, mae)
    return mae

run_te(False, 'raw14+bmi 라벨인코딩')
run_te(True, 'raw14+bmi 타겟인코딩')

raw14+bmi 라벨인코딩 0.1625
raw14+bmi 타겟인코딩 0.1616


0.1616 = 지금까지 최고 기록. 제출 파일 생성.

In [4]:
import os

test = pd.read_csv('../data/test.csv')
for col in ['medical_history', 'family_medical_history']:
    test[col] = test[col].fillna('None')
test['edu_level'] = test['edu_level'].fillna('Unknown')
test['bmi'] = test['weight'] / ((test['height'] / 100) ** 2)

train_final = pd.concat([train[['ID']], train[RAW14_BMI], train[['stress_score']]], axis=1)
test_final = pd.concat([test[['ID']], test[RAW14_BMI]], axis=1)

oof_final = np.zeros(len(train_final))
test_pred = np.zeros(len(test_final))

for tr_idx, va_idx in kf.split(train_final):
    tr_df = train_final.iloc[tr_idx].copy()
    va_df = train_final.iloc[va_idx].copy()
    te_df = test_final.copy()
    combo_tr = tr_df['medical_history'] + '_' + tr_df['family_medical_history']
    gmean = tr_df['stress_score'].mean()
    means = tr_df.groupby(combo_tr)['stress_score'].mean()
    tr_df['disease_combo_te'] = combo_tr.map(means).fillna(gmean)
    for d in (va_df, te_df):
        combo = d['medical_history'] + '_' + d['family_medical_history']
        d['disease_combo_te'] = combo.map(means).fillna(gmean)
    for d in (tr_df, va_df, te_df):
        d.drop(columns=['medical_history', 'family_medical_history'], inplace=True)
    for c in ['gender', 'activity', 'smoke_status', 'sleep_pattern', 'edu_level']:
        le = LabelEncoder().fit(tr_df[c])
        tr_df[c] = le.transform(tr_df[c])
        for d in (va_df, te_df):
            unseen = [l for l in np.unique(d[c]) if l not in le.classes_]
            if unseen:
                le.classes_ = np.append(le.classes_, unseen)
            d[c] = le.transform(d[c])
    x_tr = tr_df.drop(columns=['ID', 'stress_score']); y_tr = tr_df['stress_score']
    x_va = va_df.drop(columns=['ID', 'stress_score']); y_va = va_df['stress_score']
    x_te = te_df.drop(columns=['ID'])
    m = LGBMRegressor(**tuned_params)
    m.fit(x_tr, y_tr, eval_set=[(x_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_final[va_idx] = m.predict(x_va)
    test_pred += np.clip(m.predict(x_te), 0, 1) / kf.n_splits

print('CV:', mean_absolute_error(train_final['stress_score'], oof_final))

sample_submission = pd.read_csv('../data/sample_submission.csv')
os.makedirs('../submissions', exist_ok=True)
sample_submission['stress_score'] = np.clip(np.round(test_pred, 2), 0, 1)
sample_submission.to_csv('../submissions/submit_19_raw14_bmi_te.csv', index=False)
print('저장 완료')

CV: 0.1616
저장 완료
